# 2. Built-in AI governance and visibility

Use the same three settings as notebook 1: `AI_GATEWAY_CHAT_URL`, `AI_GATEWAY_API_KEY`, and `AI_GATEWAY_MODEL_ALIAS` in the root `.env` or shell environment. Shell values take precedence. This notebook sends at most **nine** short requests. Optional cells below can enable and remove a temporary token limit when you opt in with `use_temporary_policy`. Otherwise all requests may succeed. A backend Foundry quota failure is not proof of a gateway policy.

**Expected result:** Each request records its HTTP status and any reported token counts. Compare these with gateway policy and telemetry. A `429` requires checking its source. The optional limit applies across the gateway, so use an isolated non-production gateway.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import dotenv_values

notebook_path = globals().get('__vsc_ipynb_file__')
search_roots = [Path.cwd()]
if notebook_path:
    search_roots.insert(0, Path(notebook_path).parent)
lab_root = next(
    (directory for start in search_roots for directory in (start, *start.parents)
     if (directory / '.env.example').is_file() and (directory / 'notebooks').is_dir()),
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
)
env_path = lab_root / '.env'
file_settings = dotenv_values(env_path, encoding='utf-8-sig') if env_path.exists() else {}
required = ('AI_GATEWAY_CHAT_URL', 'AI_GATEWAY_API_KEY', 'AI_GATEWAY_MODEL_ALIAS')
settings = {name: (os.environ[name] if name in os.environ else (file_settings.get(name) or '')).strip()
            for name in required}

chat_url, api_key, alias = (settings[name] for name in required)
missing = [name for name in required if not settings[name]]
del file_settings, settings
if missing:
    status = 'found' if env_path.is_file() else 'not found'
    blank_overrides = [name for name in missing if name in os.environ]
    hint = (f" Blank kernel environment variables override .env: {', '.join(blank_overrides)}."
            ' Remove them or restart the kernel after changing the launching environment.'
            if blank_overrides else '')
    raise RuntimeError(f"Missing or blank settings: {', '.join(missing)}. "
                       f'Checked {env_path} ({status}). Set the missing values there or in the kernel environment.' + hint)
parsed = urlparse(chat_url)
if parsed.scheme != 'https' or not parsed.netloc or not parsed.path.endswith('/chat/completions'):
    raise ValueError('Use the full HTTPS Chat completions endpoint copied from the model Overview in AI Gateway.')
print('Testing alias:', alias, 'via:', parsed.hostname)


## Optional: enable a temporary token limit

Set `use_temporary_policy` to `False` to skip policy changes, or `True` to enable this exercise. Sign in with Azure CLI (`az login`) and select the subscription containing the gateway (`az account set --subscription '<subscription-id>'`). Your identity needs permission to list gateways and read/update the default workspace. The next cell automatically finds the dedicated gateway whose hostname matches `AI_GATEWAY_CHAT_URL`; no resource ID prompt or additional `.env` setting is needed. No match or multiple matches cause an error rather than guessing.

**This limit applies across the entire gateway's default workspace**, affecting all models and callers. Use an isolated non-production gateway. Existing workspace token limits are not replaced, and other policies are preserved. After enabling, allow time for the policy to propagate before running the request cell. The limit has no automatic expiration: run the cleanup cell at the end, even if an earlier cell fails. See the [README](../README.md#temporarily-enable-a-lab-token-limit-optional) for recovery and details.

In [ ]:
use_temporary_policy = False  # Set True to enable the optional gateway-wide limit.
tokens_per_minute = 1


In [ ]:
if use_temporary_policy:
    import subprocess
    import sys

    policy_result = subprocess.run(
        [sys.executable, str(lab_root / 'scripts' / 'lab_policy.py'), 'enable',
         '--chat-url', chat_url,
         '--tokens-per-minute', str(tokens_per_minute)],
        capture_output=True, text=True, check=False,
    )
    policy_exit_code = policy_result.returncode
    policy_output = policy_result.stdout + policy_result.stderr
    del policy_result
    if policy_output:
        print(policy_output, end='' if policy_output.endswith('\n') else '\n')
    if policy_exit_code:
        raise RuntimeError('Policy setup failed. If cleanup state exists, run the cleanup cell below.')
else:
    print('Temporary policy setup skipped.')


## Measure request usage and observe an optional limit

Successful responses may include provider-reported token usage. A `429` *may* indicate a gateway consumption limit, a provider quota, or another rate control: inspect the portal policy and telemetry to identify its source. Other errors are surfaced rather than mislabeled as policy enforcement.

In [ ]:
observations = []
for number in range(1, 10):
    response = requests.post(
        chat_url,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json={'model': alias, 'messages': [{'role': 'user', 'content': 'Reply with one word: hello.'}]},
        timeout=90,
    )
    if response.status_code not in (200, 429):
        response.raise_for_status()
    usage = response.json().get('usage', {}) if response.ok else {}
    observations.append({'request': number, 'status': response.status_code,
                         'prompt_tokens': usage.get('prompt_tokens'),
                         'completion_tokens': usage.get('completion_tokens')})
    print(observations[-1])
    if response.status_code == 429:
        print('Limit/quota reached; check gateway policy and backend quota before attributing the cause.')
        break


## Review policy and telemetry, then remove the temporary limit

Open your gateway's policy/consumption view and telemetry dashboard. Compare request count and token usage for this alias to the observations above; configure a telemetry destination **before** relying on historical usage. Review estimated model-spend controls, remembering that these estimates differ from actual provider billing. The README's **Concepts and scope** section explains how this management model differs from traditional API Management.

If you enabled the temporary policy, run the cleanup cell below, **even if a request or setup cell failed**. A failed cell can stop Run All before cleanup. After a kernel restart, set `cleanup_previous_policy = True` in the cleanup cell and run it without rerunning setup. Keep the Git-ignored `.lab-policy-state.json` until cleanup succeeds; it identifies the exact policy target to clean up. Allow removal to propagate before notebook 3. If you configured a limit manually, remove it in the portal instead.

In [ ]:
cleanup_previous_policy = False  # Set True to recover after a kernel restart or prior interrupted run.

if globals().get('use_temporary_policy', False) or cleanup_previous_policy:
    from pathlib import Path
    import subprocess
    import sys

    cleanup_roots = [Path.cwd()]
    cleanup_notebook = globals().get('__vsc_ipynb_file__')
    if cleanup_notebook:
        cleanup_roots.insert(0, Path(cleanup_notebook).parent)
    policy_script = next(
        (directory / 'scripts' / 'lab_policy.py'
         for start in cleanup_roots for directory in (start, *start.parents)
         if (directory / '.env.example').is_file() and (directory / 'scripts' / 'lab_policy.py').is_file()),
        None,
    )
    if policy_script is None:
        raise RuntimeError('Could not find the policy helper. Open the repository or set the kernel working directory to its root.')
    policy_result = subprocess.run(
        [sys.executable, str(policy_script), 'disable'], capture_output=True, text=True, check=False,
    )
    policy_output = policy_result.stdout + policy_result.stderr
    if policy_output:
        print(policy_output, end='' if policy_output.endswith('\n') else '\n')
    if policy_result.returncode:
        raise RuntimeError('Policy cleanup failed. Keep the local cleanup state and resolve the error above before retrying.')
else:
    print('Temporary policy cleanup skipped.')
